# APSP — Phase 2: Model Implementation

The Allocation and Periodic Scheduling Problem (APSP) assigns tasks to suitable resources and schedules their non-preemptive repetitions at a common period. This Phase 2 milestone uses the bounded monolithic MILP to minimize that period while respecting dependencies and resource capacity.

Based on van Os et al., *Optimal Resource Allocation and Periodic Scheduling*, Figure 3, Example 1, Table I, Theorem 1, and Section VI of `Paper 5.pdf`.

## Imports

In [1]:
from docplex.mp.model import Model

## Sets and data

### Example 1 - From the paper

Figure 3 and Table I give the dependencies. In $(t_1,n,t_2)$, depth $n$ means that repetition $k$ of $t_1$ must finish before repetition $k+n$ of $t_2$ can start. Example 1 gives the resource and execution time for each task. We use these as the defined entries of the partial function $e$, since the example supplies no times for other task-resource combinations.

In [2]:
T = ["x", "y", "z"]
R = ["r1", "r2"]
D = [("x", 0, "y"), ("y", 0, "z"), ("y", 3, "x")]

e = {
    ("x", "r1"): 3,
    ("y", "r2"): 3,
    ("z", "r1"): 2,
}

paper = {
    "name": "Example 1 - From the paper",
    "T": T, "R": R, "D": D, "e": e,
    "expected_period": 5,
    "expected_allocation": {"x": "r1", "y": "r2", "z": "r1"},
}

### Example 2 - Synthetic resource-allocation test

Both tasks can use either resource, and there are no dependencies. $R1$ is faster for both tasks. Assigning both to $R1$ requires a period of at least $2+2=4$. Assigning $A$ to $R2$ and $B$ to $R1$ gives a period of 3. Since using $R2$ requires at least 3 time units, this is optimal.

In [3]:
T = ["A", "B"]
R = ["R1", "R2"]
D = []

e = {
    ("A", "R1"): 2,
    ("A", "R2"): 3,
    ("B", "R1"): 2,
    ("B", "R2"): 8,
}

synthetic = {
    "name": "Example 2 - Synthetic test",
    "T": T, "R": R, "D": D, "e": e,
    "expected_period": 3,
    "expected_allocation": {"A": "R2", "B": "R1"},
}

examples = [paper, synthetic]

$R_t$ lists the suitable resources for task $t$. The ordered pairs in $O$ have distinct tasks and at least one common resource in $R_{t_1,t_2}$. With $B=|T|+|R|$, the index set $I=\{0,\ldots,B\}$ includes the extra repetition at the boundary. Repetitions start at index zero.

Each code cell below applies the same model step to both examples. Each example dictionary holds its data, model, and decision variables.

In [4]:
for example in examples:
    T = example["T"]
    R = example["R"]
    e = example["e"]

    R_t = {}
    for t in T:
        R_t[t] = [r for r in R if (t, r) in e]

    O = []
    R_shared = {}
    for t1 in T:
        for t2 in T:
            if t1 == t2:
                continue
            shared = [r for r in R_t[t1] if r in R_t[t2]]
            if shared:
                O.append((t1, t2))
                R_shared[t1, t2] = shared

    B = len(T) + len(R)
    example["R_t"] = R_t
    example["O"] = O
    example["R_shared"] = R_shared
    example["B"] = B
    example["I"] = list(range(B + 1))

**Period ceiling and Big-M.** Executing $x$, $y$, and $z$ serially at times $0$, $3$, and $6$ gives a feasible period $U=8$. The depth-three feedback edge is satisfied because $y$ finishes by the end of the period and $x$'s dependent repetition is three periods later. The minimum is therefore at most $U$.

Big-M relaxes an ordering inequality whenever the tasks do not share the resource or the opposite ordering is selected. With $\mu\le U$, $0\le s_t\le B\mu$, and $k\le B$, the largest difference Big-M must cover in (15) is at most $2BU+\max e$. In (16) it is at most $BU+\max e$. We also meet the paper's stated minimum $|T|B\max e$.

The same ceiling calculation uses $U=\sum_{t\in T}\max_{r\in R_t}e(t,r)$. For the synthetic instance below, serial execution is also feasible.

In [5]:
for example in examples:
    T = example["T"]
    e = example["e"]
    R_t = example["R_t"]
    B = example["B"]

    U = sum(max(e[t, r] for r in R_t[t]) for t in T)
    e_max = max(e.values())
    M = max(2 * B * U + e_max, len(T) * B * e_max)
    example["U"] = U
    example["M"] = M

## Decision variables

- $x_{t,r}$ is binary and equals 1 if task $t$ uses resource $r$.
- $y_{t_1,t_2,k}$ is binary. When $t_1$ and $t_2$ share a resource, it equals 1 if repetition $k$ of $t_1$ finishes before or at the start of repetition 0 of $t_2$. A value of 0 selects the reverse order.
- $s_t\ge 0$ is the initial start time. Repetition $k$ starts at $s_t+k\mu$.
- $\mu\ge 0$ is the common period. We use the continuous domain for $\mu$ and $s_t$ specified in Section VI. Definition 3 describes $\mu$ as a natural number.

In [6]:
for example in examples:
    m = Model(example["name"])

    y_keys = []
    for t1, t2 in example["O"]:
        for k in example["I"]:
            y_keys.append((t1, t2, k))

    x = m.binary_var_dict(example["e"].keys(), name="x")
    y = m.binary_var_dict(y_keys, name="y")
    s = m.continuous_var_dict(example["T"], lb=0, name="s")
    mu = m.continuous_var(lb=0, ub=example["U"], name="mu")

    example["m"] = m
    example["x"] = x
    example["y"] = y
    example["s"] = s
    example["mu"] = mu

## Objective function

$$\min_{\mu,s,x,y}\ \mu \tag{12}$$

Minimize the common period to maximize throughput.

In [7]:
for example in examples:
    m = example["m"]
    mu = example["mu"]
    m.minimize(mu)

## Constraints

### (13) Assign each task to one suitable resource

$$\sum_{r\in R_t} x_{t,r}=1. \tag{13}$$

For every $t\in T$.

Exactly one allocation variable is selected for each task.

In [8]:
for example in examples:
    m = example["m"]
    x = example["x"]
    R_t = example["R_t"]
    for t in example["T"]:
        m.add_constraint(m.sum(x[t, r] for r in R_t[t]) == 1)

### (14) Respect task dependencies

$$
\begin{aligned}
s_{t_2} \ge{} & s_{t_1}-n\mu \\
&+\sum_{r\in R_{t_1}} x_{t_1,r}e(t_1,r).
\end{aligned}
\tag{14}
$$

For every $(t_1,n,t_2)\in D$.

The successor's repetition $n$ must start at or after the predecessor's initial completion. Adding $k\mu$ to both sides gives the same rule for any predecessor repetition $k$.

In [9]:
for example in examples:
    m = example["m"]
    x = example["x"]
    s = example["s"]
    mu = example["mu"]
    e = example["e"]
    R_t = example["R_t"]
    for t1, n, t2 in example["D"]:
        m.add_constraint(
            s[t2] >= s[t1] - n * mu + m.sum(x[t1, r] * e[t1, r] for r in R_t[t1])
        )

### (15) Order $t_1$ before $t_2$ on a shared resource

$$
\begin{aligned}
&s_{t_1}+e(t_1,r)+k\mu \\
&\quad\le s_{t_2} \\
&\qquad+M(3-x_{t_1,r}-x_{t_2,r}-y_{t_1,t_2,k}).
\end{aligned}
\tag{15}
$$

For each $(t_1,t_2)\in O$, $r\in R_{t_1,t_2}$, and $k\in I$, this enforces the order when both tasks use $r$ and $y_{t_1,t_2,k}=1$.

In [10]:
for example in examples:
    m = example["m"]
    x = example["x"]
    y = example["y"]
    s = example["s"]
    mu = example["mu"]
    e = example["e"]
    M = example["M"]
    for t1, t2 in example["O"]:
        for r in example["R_shared"][t1, t2]:
            for k in example["I"]:
                m.add_constraint(
                    s[t1] + e[t1, r] + k * mu
                    <= s[t2] + M * (3 - x[t1, r] - x[t2, r] - y[t1, t2, k])
                )

### (16) Order $t_2$ before $t_1$ on a shared resource

$$
\begin{aligned}
&s_{t_2}+e(t_2,r) \\
&\quad\le s_{t_1}+k\mu \\
&\qquad+M(2-x_{t_1,r}-x_{t_2,r}+y_{t_1,t_2,k}).
\end{aligned}
\tag{16}
$$

For the same indices, this enforces the reverse order when both tasks use $r$ and $y_{t_1,t_2,k}=0$. Together, (15) and (16) prevent overlap between distinct tasks.

In [11]:
for example in examples:
    m = example["m"]
    x = example["x"]
    y = example["y"]
    s = example["s"]
    mu = example["mu"]
    e = example["e"]
    M = example["M"]
    for t1, t2 in example["O"]:
        for r in example["R_shared"][t1, t2]:
            for k in example["I"]:
                m.add_constraint(
                    s[t2] + e[t2, r]
                    <= s[t1] + k * mu + M * (2 - x[t1, r] - x[t2, r] + y[t1, t2, k])
                )

### (17) Prevent overlap between repetitions of the same task

$$
\begin{aligned}
&s_t+\sum_{r\in R_t} x_{t,r}e(t,r)\le s_t+\mu.
\end{aligned}
\tag{17}
$$

For every $t\in T$.

The initial repetition must finish by the next start. The same separation then holds between all consecutive repetitions.

In [12]:
for example in examples:
    m = example["m"]
    x = example["x"]
    s = example["s"]
    mu = example["mu"]
    e = example["e"]
    R_t = example["R_t"]
    for t in example["T"]:
        m.add_constraint(s[t] + m.sum(x[t, r] * e[t, r] for r in R_t[t]) <= s[t] + mu)

### (18) Bound the initial start times

$$s_t\le B\mu. \tag{18}$$

For every $t\in T$.

Theorem 1 guarantees that a feasible schedule can be shifted to satisfy $s_t<B\mu$ at the same period. The model uses the closed inequality printed in (18). For $k>B$, $s_{t_1}+k\mu\ge (B+1)\mu$, by which time every initial repetition has finished by (17) and (18). Both pair directions and the common period extend the finite overlap checks to all repetitions.

In [13]:
for example in examples:
    m = example["m"]
    s = example["s"]
    mu = example["mu"]
    B = example["B"]
    for t in example["T"]:
        m.add_constraint(s[t] <= B * mu)

## Solve

Solve both examples using the Monolithic model.

In [14]:
for example in examples:
    m = example["m"]
    solution = m.solve()
    if solution is None:
        raise RuntimeError(f"CPLEX returned no schedule for {example['name']}. Status: {m.solve_details.status}")
    example["solution"] = solution
    print(f"{example['name']} - CPLEX status: {m.solve_details.status}")

Example 1 - From the paper - CPLEX status: integer optimal solution
Example 2 - Synthetic test - CPLEX status: integer optimal solution


## Results and basic verification

Display and verify both solutions using the solver's actual values. Check allocation binaries, dependencies, consecutive repetitions, the start-time bound, and shared-resource overlap across repetitions. The overlap checks use actual start and finish times independently of Big-M and $y$. Both task-pair directions and $k=0,\ldots,B$ cover all relative repetitions under (17) and (18).

For the paper instance, equation (7) gives a lower bound of $3+2=5$ from $x$ and $z$ on $r_1$. Initial start times may differ between valid optimal schedules. A failed check reports the violated condition.

In [15]:
tol = 1e-6

for example in examples:
    print(f"\n{example['name']}")
    T = example["T"]
    R = example["R"]
    D = example["D"]
    e = example["e"]
    solution = example["solution"]
    x = example["x"]
    s = example["s"]
    mu = example["mu"]
    period = solution.get_value(mu)
    B = len(T) + len(R)
    start = {}
    allocation = {}
    duration = {}

    assert period > 0, f"Nonpositive period {period}"
    print(f"Period (mu): {period:g}")
    print(f"{'Task':<6}{'Resource':<10}{'Initial start':>14}")
    for t in T:
        R_t = [r for r in R if (t, r) in e]
        values = [solution.get_value(x[t, r]) for r in R_t]
        assert all(min(abs(value), abs(value - 1)) <= tol for value in values), (
            f"Nonbinary allocation for {t}"
        )
        assert abs(sum(values) - 1) <= tol, f"Allocation sum violated for {t}"
        assigned = [r for r in R_t if solution.get_value(x[t, r]) > 0.5]
        assert len(assigned) == 1, f"Invalid resource allocation for {t}"
        allocation[t] = assigned[0]
        start[t] = solution.get_value(s[t])
        duration[t] = e[t, allocation[t]]
        assert duration[t] <= period + tol, f"Consecutive repetitions overlap for {t}"
        assert -tol <= start[t] <= B * period + tol, f"Start bound violated for {t}"
        print(f"{t:<6}{allocation[t]:<10}{start[t]:>14g}")

    for t1, n, t2 in D:
        assert start[t1] + duration[t1] <= start[t2] + n * period + tol, (
            f"Dependency violated for {(t1, n, t2)}"
        )

    for t1 in T:
        for t2 in T:
            if t1 != t2 and allocation[t1] == allocation[t2]:
                for k in range(B + 1):
                    repetition_start = start[t1] + k * period
                    t1_first = repetition_start + duration[t1] <= start[t2] + tol
                    t2_first = start[t2] + duration[t2] <= repetition_start + tol
                    assert t1_first or t2_first, (
                        f"Overlap between {t1}[{k}] and {t2}[0]"
                    )

    print("All constraints satisfied.")


Example 1 - From the paper
Period (mu): 5
Task  Resource   Initial start
x     r1                     0
y     r2                     3
z     r1                    23
All constraints satisfied.

Example 2 - Synthetic test
Period (mu): 3
Task  Resource   Initial start
A     R2                     0
B     R1                     0
All constraints satisfied.


In [16]:
# Check the expected objective after schedule feasibility has been verified.
for example in examples:
    period = example["solution"].get_value(example["mu"])
    assert abs(period - example["expected_period"]) <= tol, "Unexpected Monolithic period"

# Benders decomposition | Part 1

This section solves one master problem and one scheduling subproblem for each example. The subproblem minimizes the period for the allocation and ordering selected by the master. With an empty cut set, that period can differ from the global APSP optimum.

## Master problem (MP)

The binary variables $x_{t,r}$ and $y_{t_1,t_2,k}$ have the same meanings as in the Monolithic model. The new continuous variables are $z\ge 0$, a lower bound on the optimal period, and $w_t\ge 0$, a dummy initial start time for task $t$.

The paper fixes the period in the master's scheduling constraints at $\hat{\mu}=\sum_{t\in T}\max_{r\in R_t}e(t,r)$. This is the ceiling $U$ already computed above. At this fixed period, $w$ helps screen allocations and orderings for timing consistency. The master minimizes $z$, while the subproblem finds the shortest period for the chosen decisions.

In [17]:
for example in examples:
    mp = Model("Master - " + example["name"])
    y_keys = []
    for t1, t2 in example["O"]:
        for k in example["I"]:
            y_keys.append((t1, t2, k))

    example["mp"] = mp
    example["x_mp"] = mp.binary_var_dict(example["e"].keys(), name="x")
    example["y_mp"] = mp.binary_var_dict(y_keys, name="y")
    example["z"] = mp.continuous_var(lb=0, name="z")
    example["w"] = mp.continuous_var_dict(example["T"], lb=0, name="w")

### (19) Minimize the lower bound

$$\min_{x,y,z,w}\ z. \tag{19}$$

In [18]:
for example in examples:
    mp = example["mp"]
    mp.minimize(example["z"])

### (20) Assign each task to one suitable resource

$$\sum_{r\in R_t}x_{t,r}=1. \tag{20}$$

Apply this constraint to every task $t\in T$.

In [19]:
for example in examples:
    mp = example["mp"]
    x_mp = example["x_mp"]
    R_t = example["R_t"]
    for t in example["T"]:
        mp.add_constraint(mp.sum(x_mp[t, r] for r in R_t[t]) == 1)

### (21) Cut bounds

$$
\begin{aligned}
&z\ge c,\qquad c\in C.
\end{aligned}
\tag{21}
$$

The cut set $C$ is empty in Part 1, so (21) adds zero constraints. The minimum of $z$ is therefore zero.

### (22) Respect dependencies at the period ceiling

$$
\begin{aligned}
w_{t_2}\ge{}&w_{t_1}-n\hat{\mu}\\
&+\sum_{r\in R_{t_1}}x_{t_1,r}e(t_1,r).
\end{aligned}
\tag{22}
$$

For every $(t_1,n,t_2)\in D$, enforce precedence using the dummy starts and the fixed period.

In [20]:
for example in examples:
    mp = example["mp"]
    x_mp = example["x_mp"]
    w = example["w"]
    mu_hat = example["U"]
    e = example["e"]
    R_t = example["R_t"]
    for t1, n, t2 in example["D"]:
        mp.add_constraint(
            w[t2] >= w[t1] - n * mu_hat + mp.sum(x_mp[t1, r] * e[t1, r] for r in R_t[t1])
        )

### (23) Order repetition $k$ of $t_1$ before $t_2$

$$
\begin{aligned}
&w_{t_1}+e(t_1,r)+k\hat{\mu}\\
&\quad\le w_{t_2}\\
&\qquad+M(3-x_{t_1,r}-x_{t_2,r}-y_{t_1,t_2,k}).
\end{aligned}
\tag{23}
$$

For $(t_1,t_2)\in O$, $r\in R_{t_1,t_2}$, and $k\in I$, this order applies when both tasks use $r$ and $y_{t_1,t_2,k}=1$.

In [21]:
for example in examples:
    mp = example["mp"]
    x_mp = example["x_mp"]
    y_mp = example["y_mp"]
    w = example["w"]
    mu_hat = example["U"]
    e = example["e"]
    M = example["M"]
    for t1, t2 in example["O"]:
        for r in example["R_shared"][t1, t2]:
            for k in example["I"]:
                mp.add_constraint(
                    w[t1] + e[t1, r] + k * mu_hat
                    <= w[t2] + M * (3 - x_mp[t1, r] - x_mp[t2, r] - y_mp[t1, t2, k])
                )

### (24) Order $t_2$ before repetition $k$ of $t_1$

$$
\begin{aligned}
&w_{t_2}+e(t_2,r)\\
&\quad\le w_{t_1}+k\hat{\mu}\\
&\qquad+M(2-x_{t_1,r}-x_{t_2,r}+y_{t_1,t_2,k}).
\end{aligned}
\tag{24}
$$

For the same indices, the reverse order applies when both tasks use $r$ and $y_{t_1,t_2,k}=0$.

In [22]:
for example in examples:
    mp = example["mp"]
    x_mp = example["x_mp"]
    y_mp = example["y_mp"]
    w = example["w"]
    mu_hat = example["U"]
    e = example["e"]
    M = example["M"]
    for t1, t2 in example["O"]:
        for r in example["R_shared"][t1, t2]:
            for k in example["I"]:
                mp.add_constraint(
                    w[t2] + e[t2, r]
                    <= w[t1] + k * mu_hat
                    + M * (2 - x_mp[t1, r] - x_mp[t2, r] + y_mp[t1, t2, k])
                )

### (25) Separate consecutive repetitions of each task

$$
\begin{aligned}
&w_t+\sum_{r\in R_t}x_{t,r}e(t,r)\le w_t+\hat{\mu}.
\end{aligned}
\tag{25}
$$

For every task $t\in T$, the assigned execution time must fit within the fixed period.

In [23]:
for example in examples:
    mp = example["mp"]
    x_mp = example["x_mp"]
    w = example["w"]
    mu_hat = example["U"]
    e = example["e"]
    R_t = example["R_t"]
    for t in example["T"]:
        mp.add_constraint(w[t] + mp.sum(x_mp[t, r] * e[t, r] for r in R_t[t]) <= w[t] + mu_hat)

### Solve the master and fix the trial decisions

Read $\bar{x}$ and $\bar{y}$ from the master solution as integers. The subproblem uses these decisions as constants and computes its own initial starts.

In [24]:
def read_binary_values(solution, variables):
    values = {}
    for key, variable in variables.items():
        value = solution.get_value(variable)
        assert min(abs(value), abs(value - 1)) <= tol, f"Nonbinary value for {variable.name}: {value}"
        values[key] = round(value)
    return values


for example in examples:
    mp = example["mp"]
    mp_solution = mp.solve()
    print(f"{example['name']} - MP status: {mp.solve_details.status}")
    if mp_solution is None:
        raise RuntimeError(f"No master solution for {example['name']}")

    x_bar = read_binary_values(mp_solution, example["x_mp"])
    y_bar = read_binary_values(mp_solution, example["y_mp"])

    example["x_bar"] = x_bar
    example["y_bar"] = y_bar
    print(f"Master lower bound z: {mp_solution.get_value(example['z']):g}")
    for (t, r), assigned in x_bar.items():
        if assigned == 1:
            print(f"  {t} -> {r}")

Example 1 - From the paper - MP status: integer optimal solution
Master lower bound z: 0
  x -> r1
  y -> r2
  z -> r1
Example 2 - Synthetic test - MP status: integer optimal solution
Master lower bound z: 0
  A -> R2
  B -> R2


## Scheduling subproblem (SP)

The only decision variables are the common period $\mu\ge 0$ and the initial start times $s_t\ge 0$. With $\bar{x}$ and $\bar{y}$ fixed, the scheduling subproblem is a linear program.

### (26) Minimize the period for the trial decisions

$$
\begin{aligned}
&\min_{\mu,s}\ \mu.
\end{aligned}
\tag{26}
$$

In [25]:
for example in examples:
    sp = Model("Scheduling - " + example["name"])
    mu_sp = sp.continuous_var(lb=0, name="mu")
    s_sp = sp.continuous_var_dict(example["T"], lb=0, name="s")
    sp.minimize(mu_sp)

    example["sp"] = sp
    example["mu_sp"] = mu_sp
    example["s_sp"] = s_sp

### (27) Respect dependencies with the fixed allocation

$$
\begin{aligned}
s_{t_2}\ge{}&s_{t_1}-n\mu\\
&+\sum_{r\in R_{t_1}}\bar{x}_{t_1,r}e(t_1,r).
\end{aligned}
\tag{27}
$$

For every $(t_1,n,t_2)\in D$, the fixed allocation gives the predecessor's duration.

In [26]:
for example in examples:
    sp = example["sp"]
    x_bar = example["x_bar"]
    s_sp = example["s_sp"]
    mu_sp = example["mu_sp"]
    e = example["e"]
    R_t = example["R_t"]
    for t1, n, t2 in example["D"]:
        sp.add_constraint(
            s_sp[t2] >= s_sp[t1] - n * mu_sp + sp.sum(x_bar[t1, r] * e[t1, r] for r in R_t[t1])
        )

### (28) Enforce the forward order with fixed decisions

$$
\begin{aligned}
&s_{t_1}+e(t_1,r)+k\mu\\
&\quad\le s_{t_2}\\
&\qquad+M(3-\bar{x}_{t_1,r}-\bar{x}_{t_2,r}-\bar{y}_{t_1,t_2,k}).
\end{aligned}
\tag{28}
$$

Use $(t_1,t_2)\in O$, $r\in R_{t_1,t_2}$, and $k\in I$. The printed index $m\in I$ refers to the repetition index $k$ used in the inequality.

In [27]:
for example in examples:
    sp = example["sp"]
    x_bar = example["x_bar"]
    y_bar = example["y_bar"]
    s_sp = example["s_sp"]
    mu_sp = example["mu_sp"]
    e = example["e"]
    M = example["M"]
    for t1, t2 in example["O"]:
        for r in example["R_shared"][t1, t2]:
            for k in example["I"]:
                sp.add_constraint(
                    s_sp[t1] + e[t1, r] + k * mu_sp
                    <= s_sp[t2] + M * (3 - x_bar[t1, r] - x_bar[t2, r] - y_bar[t1, t2, k])
                )

### (29) Enforce the reverse order with fixed decisions

$$
\begin{aligned}
&s_{t_2}+e(t_2,r)\\
&\quad\le s_{t_1}+k\mu\\
&\qquad+M(2-\bar{x}_{t_1,r}-\bar{x}_{t_2,r}+\bar{y}_{t_1,t_2,k}).
\end{aligned}
\tag{29}
$$

The fixed value of $\bar{y}_{t_1,t_2,k}$ determines which of (28) and (29) applies when both tasks use $r$.

In [28]:
for example in examples:
    sp = example["sp"]
    x_bar = example["x_bar"]
    y_bar = example["y_bar"]
    s_sp = example["s_sp"]
    mu_sp = example["mu_sp"]
    e = example["e"]
    M = example["M"]
    for t1, t2 in example["O"]:
        for r in example["R_shared"][t1, t2]:
            for k in example["I"]:
                sp.add_constraint(
                    s_sp[t2] + e[t2, r]
                    <= s_sp[t1] + k * mu_sp
                    + M * (2 - x_bar[t1, r] - x_bar[t2, r] + y_bar[t1, t2, k])
                )

### (30) Separate consecutive repetitions

$$
\begin{aligned}
&s_t+\sum_{r\in R_t}\bar{x}_{t,r}e(t,r)\le s_t+\mu.
\end{aligned}
\tag{30}
$$

For every task $t\in T$, the fixed duration must fit within the period.

In [29]:
for example in examples:
    sp = example["sp"]
    x_bar = example["x_bar"]
    s_sp = example["s_sp"]
    mu_sp = example["mu_sp"]
    e = example["e"]
    R_t = example["R_t"]
    for t in example["T"]:
        sp.add_constraint(s_sp[t] + sp.sum(x_bar[t, r] * e[t, r] for r in R_t[t]) <= s_sp[t] + mu_sp)

### (31) Bound the initial starts

$$s_t\le B\mu. \tag{31}$$

Apply the same bounded start condition as in (18) to each task $t\in T$.

In [30]:
for example in examples:
    sp = example["sp"]
    s_sp = example["s_sp"]
    mu_sp = example["mu_sp"]
    B = example["B"]
    for t in example["T"]:
        sp.add_constraint(s_sp[t] <= B * mu_sp)

### Solve and verify the fixed schedule

Check the selected resources, dependency timing, start bounds, and non-overlap directly from the returned schedule. The minimum period reported here is conditional on the master decisions.

In [31]:
for example in examples:
    print(f"\n{example['name']}")
    sp = example["sp"]
    sp_solution = sp.solve()
    print(f"SP status: {sp.solve_details.status}")
    if sp_solution is None:
        raise RuntimeError(f"No scheduling solution for {example['name']}")

    T = example["T"]
    e = example["e"]
    B = example["B"]
    x_bar = example["x_bar"]
    period = sp_solution.get_value(example["mu_sp"])
    allocation = {}
    start = {}
    duration = {}

    assert period > 0, f"Nonpositive period {period}"
    print(f"Minimum period for fixed decisions: {period:g}")
    print(f"{'Task':<6}{'Resource':<10}{'Initial start':>14}")
    for t in T:
        assigned = [r for r in example["R_t"][t] if x_bar[t, r] == 1]
        assert len(assigned) == 1, f"Invalid resource allocation for {t}"
        allocation[t] = assigned[0]
        start[t] = sp_solution.get_value(example["s_sp"][t])
        duration[t] = e[t, allocation[t]]
        assert -tol <= start[t] <= B * period + tol, f"Start bound violated for {t}"
        assert duration[t] <= period + tol, f"Consecutive repetitions overlap for {t}"
        print(f"{t:<6}{allocation[t]:<10}{start[t]:>14g}")

    for t1, n, t2 in example["D"]:
        assert start[t1] + duration[t1] <= start[t2] + n * period + tol, (
            f"Dependency violated for {(t1, n, t2)}"
        )

    for t1, t2 in example["O"]:
        if allocation[t1] == allocation[t2]:
            for k in example["I"]:
                repetition_start = start[t1] + k * period
                t1_first = repetition_start + duration[t1] <= start[t2] + tol
                t2_first = start[t2] + duration[t2] <= repetition_start + tol
                assert t1_first or t2_first, f"Overlap between {t1}[{k}] and {t2}[0]"

    print("APSP constraints satisfied: Yes")


Example 1 - From the paper
SP status: optimal
Minimum period for fixed decisions: 8
Task  Resource   Initial start
x     r1                     0
y     r2                     3
z     r1                     6
APSP constraints satisfied: Yes

Example 2 - Synthetic test
SP status: optimal
Minimum period for fixed decisions: 11
Task  Resource   Initial start
A     R2                     0
B     R2                    36
APSP constraints satisfied: Yes


# Benders decomposition | Part 2

## Dual scheduling subproblem (DSP)

Each nonnegative dual variable weights one scheduling constraint. The families $\pi^p,\pi^{nl},\pi^{nr},\pi^{nd},\pi^s$ correspond to dependencies (27), forward and reverse orderings (28)–(29), consecutive repetitions (30), and start bounds (31).

A feasible set of weights gives a lower bound on the minimum period for fixed $\bar x,\bar y$. Strong LP duality makes the best such bound equal to the optimal SP period when the LP has a finite optimum. The same weights can give bounds for other choices of $x,y$.

### (35) Weighted scheduling bounds

Write (27)–(31) with the start times and period on the left of $\ge$. Multiply each right-hand side by its dual weight and sum. The set $J$ contains each ordered pair of tasks together with a shared resource and repetition index.

$$
\begin{aligned}
ob(\pi,x,y)={}&
\sum_{(t_1,n,t_2)\in D}\pi^p_{t_1,n,t_2}
  \sum_{r\in R_{t_1}}x_{t_1,r}e(t_1,r)\\
&+\sum_{(t_1,t_2,r,k)\in J}\pi^{nl}_{t_1,t_2,r,k}\\
&\qquad{}\cdot[e(t_1,r)-M(3-x_{t_1,r}-x_{t_2,r}-y_{t_1,t_2,k})]\\
&+\sum_{(t_1,t_2,r,k)\in J}\pi^{nr}_{t_1,t_2,r,k}\\
&\qquad{}\cdot[e(t_2,r)-M(2-x_{t_1,r}-x_{t_2,r}+y_{t_1,t_2,k})]\\
&+\sum_{t\in T}\pi^{nd}_t\sum_{r\in R_t}x_{t,r}e(t,r).
\end{aligned}
\tag{35}
$$

The start-bound rows contribute zero because their right-hand side is zero. Use this same expression for the dual objective and the cut.

In [32]:
def dual_objective(m, example, pi, x, y):
    e = example["e"]
    M = example["M"]
    duration = {}
    for t in example["T"]:
        duration[t] = m.sum(x[t, r] * e[t, r] for r in example["R_t"][t])

    terms = []
    for t1, n, t2 in example["D"]:
        terms.append(pi["p"][t1, n, t2] * duration[t1])
    for t1, t2, r, k in pi["nl"]:
        forward_rhs = e[t1, r] - M * (3 - x[t1, r] - x[t2, r] - y[t1, t2, k])
        reverse_rhs = e[t2, r] - M * (2 - x[t1, r] - x[t2, r] + y[t1, t2, k])
        terms.append(pi["nl"][t1, t2, r, k] * forward_rhs)
        terms.append(pi["nr"][t1, t2, r, k] * reverse_rhs)
    for t in example["T"]:
        terms.append(pi["nd"][t] * duration[t])
    return m.sum(terms)

### (32) Maximize the bound for the trial decisions

$$
\begin{aligned}
&\max_{\pi\ge0}\ ob(\pi,\bar x,\bar y).
\end{aligned}
\tag{32}
$$

With the trial binaries fixed, the objective is linear in the dual variables.

In [33]:
from docplex.util.status import JobSolveStatus

def make_dsp(example, x_bar, y_bar):
    dsp = Model("Dual scheduling - " + example["name"])
    J = []
    for t1, t2 in example["O"]:
        for r in example["R_shared"][t1, t2]:
            for k in example["I"]:
                J.append((t1, t2, r, k))

    pi = {
        "p": dsp.continuous_var_dict(example["D"], lb=0, name="pi_p"),
        "nl": dsp.continuous_var_dict(J, lb=0, name="pi_nl"),
        "nr": dsp.continuous_var_dict(J, lb=0, name="pi_nr"),
        "nd": dsp.continuous_var_dict(example["T"], lb=0, name="pi_nd"),
        "s": dsp.continuous_var_dict(example["T"], lb=0, name="pi_s"),
    }
    dsp.maximize(dual_objective(dsp, example, pi, x_bar, y_bar))
    return dsp, pi

### (33) Limit the total coefficient of the period

$$
\begin{aligned}
&\sum_{(t_1,n,t_2)\in D}n\pi^p_{t_1,n,t_2}\\
&+\sum_{(t_1,t_2,r,k)\in J} k(\pi^{nr}_{t_1,t_2,r,k}-\pi^{nl}_{t_1,t_2,r,k})\\
&+\sum_{t\in T}(\pi^{nd}_t+B\pi^s_t)\le1.
\end{aligned}
\tag{33}
$$

The bound of 1 is the primal objective coefficient of $\mu$. In the $\ge$ form, (28) contributes $-k$ and (29) contributes $+k$.

In [34]:
def add_dual_period_constraint(dsp, example, pi):
    period_weight = dsp.sum(n * pi["p"][t1, n, t2] for t1, n, t2 in example["D"])
    period_weight += dsp.sum(
        k * (pi["nr"][t1, t2, r, k] - pi["nl"][t1, t2, r, k])
        for t1, t2, r, k in pi["nl"]
    )
    period_weight += dsp.sum(
        pi["nd"][t] + example["B"] * pi["s"][t] for t in example["T"]
    )
    dsp.add_constraint(period_weight <= 1)

### (34) Limit the total coefficient of each start time

$$
\begin{aligned}
&\sum_{(u,n,t)\in D}\pi^p_{u,n,t}
 -\sum_{(t,n,u)\in D}\pi^p_{t,n,u}\\
&+\sum_{(t,u,r,k)\in J}(\pi^{nr}_{t,u,r,k}-\pi^{nl}_{t,u,r,k})\\
&+\sum_{(u,t,r,k)\in J}(\pi^{nl}_{u,t,r,k}-\pi^{nr}_{u,t,r,k})\\
&-\pi^s_t\le0,\qquad t\in T.
\end{aligned}
\tag{34}
$$

Each $s_t$ has coefficient zero in the primal objective. In the normalized dependency row (27), the successor has coefficient $+1$ and the predecessor has coefficient $-1$. In (28) and (29), the two start coefficients are opposite, and (31) contributes a coefficient of $-1$ for its task.

In [35]:
def add_dual_start_constraints(dsp, example, pi):
    start_weight = {t: -pi["s"][t] for t in example["T"]}
    for t1, n, t2 in example["D"]:
        start_weight[t1] -= pi["p"][t1, n, t2]
        start_weight[t2] += pi["p"][t1, n, t2]
    for t1, t2, r, k in pi["nl"]:
        weight = pi["nr"][t1, t2, r, k] - pi["nl"][t1, t2, r, k]
        start_weight[t1] += weight
        start_weight[t2] -= weight
    for t in example["T"]:
        dsp.add_constraint(start_weight[t] <= 0)

### Compare the primal and dual

Use the same master decisions as in Part 1. Check that the optimal period from SP agrees with the DSP objective.

In [36]:
def solve_dsp(example, x_bar, y_bar):
    dsp, pi = make_dsp(example, x_bar, y_bar)
    add_dual_period_constraint(dsp, example, pi)
    add_dual_start_constraints(dsp, example, pi)
    solution = dsp.solve()
    if dsp.get_solve_status() != JobSolveStatus.OPTIMAL_SOLUTION:
        raise RuntimeError(f"DSP status for {example['name']}: {dsp.solve_details.status}")
    pi_bar = {}
    for family, variables in pi.items():
        pi_bar[family] = {key: solution.get_value(var) for key, var in variables.items()}
    return solution.objective_value, pi_bar

for example in examples:
    primal_value = example["sp"].solution.objective_value
    dual_value, pi_bar = solve_dsp(example, example["x_bar"], example["y_bar"])
    assert abs(primal_value - dual_value) <= tol, "SP and DSP objectives differ"
    print(f"{example['name']} | SP = {primal_value:g} | DSP = {dual_value:g}")

Example 1 - From the paper | SP = 8 | DSP = 8
Example 2 - Synthetic test | SP = 11 | DSP = 11


## Benders optimality cut

$$
\begin{aligned}
&z\ge ob(\bar\pi,x,y).
\end{aligned}
\tag{21}
$$

Fix the weights $\bar\pi$ from the dual solution and leave $x,y$ as master variables. The dual constraints (33)–(34) do not depend on the trial binaries, so the same weights give a valid lower bound for every master choice. At the current trial, the cut's right-hand side equals its SP optimum by strong duality. The cut makes the master account for this bound in its next solve.

In [37]:
def add_benders_cut(mp, example, z, x_mp, y_mp, pi_bar, mp_solution, period):
    cut_rhs = dual_objective(mp, example, pi_bar, x_mp, y_mp)
    trial_bound = mp_solution.get_value(cut_rhs)
    assert abs(trial_bound - period) <= tol, "Cut is not tight at the trial decisions"
    mp.add_constraint(z >= cut_rhs)

## Repeat the scheduling LP for new master decisions

Use the same SP constraints (26)–(31) from Part 1 with the current values of $\bar x$ and $\bar y$. Keep the returned period and starts together.

In [38]:
def solve_sp(example, x_bar, y_bar):
    sp = Model("Scheduling trial - " + example["name"])
    T = example["T"]
    e = example["e"]
    M = example["M"]
    B = example["B"]
    mu = sp.continuous_var(lb=0, name="mu")
    s = sp.continuous_var_dict(T, lb=0, name="s")
    sp.minimize(mu)  # (26)

    duration = {}
    for t in T:
        duration[t] = sp.sum(x_bar[t, r] * e[t, r] for r in example["R_t"][t])

    for t1, n, t2 in example["D"]:  # (27)
        sp.add_constraint(s[t2] >= s[t1] - n * mu + duration[t1])

    for t1, t2 in example["O"]:
        for r in example["R_shared"][t1, t2]:
            for k in example["I"]:
                both = x_bar[t1, r] + x_bar[t2, r]
                order = y_bar[t1, t2, k]
                sp.add_constraint(  # (28)
                    s[t1] + e[t1, r] + k * mu <= s[t2] + M * (3 - both - order)
                )
                sp.add_constraint(  # (29)
                    s[t2] + e[t2, r] <= s[t1] + k * mu + M * (2 - both + order)
                )

    for t in T:
        sp.add_constraint(s[t] + duration[t] <= s[t] + mu)  # (30)
        sp.add_constraint(s[t] <= B * mu)  # (31)

    solution = sp.solve()
    if sp.get_solve_status() != JobSolveStatus.OPTIMAL_SOLUTION:
        raise RuntimeError(f"SP status for {example['name']}: {sp.solve_details.status}")
    return solution.get_value(mu), {t: solution.get_value(s[t]) for t in T}

# Benders decomposition | Part 3

A degenerate dual can have several optimal multiplier vectors. Their cuts agree at the current trial but can give different lower bounds elsewhere. The Magnanti–Wong auxiliary subproblem (ASP) keeps the cut tight at the trial and selects multipliers that give the highest bound at the Papadakos core point.

## Update the Papadakos core points

For either binary family $v=x$ or $v=y$, use the paper's recurrence, with iterations indexed from zero.

$$
\begin{aligned}
\bar v^{0,0} &= \tfrac12\bar v^0,\\
\bar v^{0,k} &= \tfrac12\bigl(\bar v^{0,k-1}+\bar v^k\bigr),\qquad k>0.
\end{aligned}
$$

An empty previous dictionary supplies zero on the first iteration. Update the two families separately.

In [39]:
def update_core(previous, trial):
    return {key: (previous.get(key, 0) + value) / 2 for key, value in trial.items()}

## Magnanti–Wong auxiliary subproblem (ASP)

### (36) Maximize the bound at the core point

$$
\begin{aligned}
&\max_{\pi\ge0}\ ob(\pi,\bar x^0,\bar y^0).
\end{aligned}
\tag{36}
$$

Reuse the dual builder with the core points in the objective and keep the same feasibility constraints (33) and (34).

In [40]:
def make_asp(example, x_core, y_core):
    asp, pi = make_dsp(example, x_core, y_core)
    add_dual_period_constraint(asp, example, pi)
    add_dual_start_constraints(asp, example, pi)
    return asp, pi

### (37) Keep the cut tight at the trial

$$
\begin{aligned}
&ob(\pi,\bar x,\bar y)=ob_{\mathrm{DSP}}.
\end{aligned}
\tag{37}
$$

Use an equality with CPLEX's LP feasibility tolerance and check the residual explicitly. The auxiliary solution must satisfy the dual constraints and match the DSP optimum at the trial. Its multipliers then enter the same symbolic cut (21).

In [41]:
def solve_asp(example, x_bar, y_bar, x_core, y_core, dual_value):
    asp, pi = make_asp(example, x_core, y_core)
    trial_objective = dual_objective(asp, example, pi, x_bar, y_bar)
    asp.add_constraint(trial_objective == dual_value)
    asp.parameters.simplex.tolerances.feasibility = 1e-9
    solution = asp.solve()
    if asp.get_solve_status() != JobSolveStatus.OPTIMAL_SOLUTION:
        raise RuntimeError(f"ASP status for {example['name']}: {asp.solve_details.status}")

    assert solution.is_valid_solution(tolerance=tol), "ASP solution violates dual feasibility or (37)"
    trial_value = solution.get_value(trial_objective)
    assert abs(trial_value - dual_value) <= tol, "ASP is not tight at the trial"
    return {
        family: {key: solution.get_value(var) for key, var in variables.items()}
        for family, variables in pi.items()
    }

### Schedule feasibility

Check a candidate schedule before it becomes the upper bound. Use the same allocation, start-bound, dependency, and overlap checks as in Part 2.

In [42]:
def verify_benders_schedule(example, period, x_bar, start):
    allocation = {}
    duration = {}

    assert period > 0, f"Nonpositive period {period}"
    for t in example["T"]:
        assigned = [r for r in example["R_t"][t] if x_bar[t, r] == 1]
        assert len(assigned) == 1, f"Invalid resource allocation for {t}"
        allocation[t] = assigned[0]
        duration[t] = example["e"][t, allocation[t]]
        assert -tol <= start[t] <= example["B"] * period + tol, f"Start bound violated for {t}"
        assert duration[t] <= period + tol, f"Consecutive repetitions overlap for {t}"

    for t1, n, t2 in example["D"]:
        assert start[t1] + duration[t1] <= start[t2] + n * period + tol, (
            f"Dependency violated for {(t1, n, t2)}"
        )
    for t1, t2 in example["O"]:
        if allocation[t1] == allocation[t2]:
            for k in example["I"]:
                repetition_start = start[t1] + k * period
                t1_first = repetition_start + duration[t1] <= start[t2] + tol
                t2_first = start[t2] + duration[t2] <= repetition_start + tol
                assert t1_first or t2_first, f"Overlap between {t1}[{k}] and {t2}[0]"
    return allocation

## Basic and accelerated Benders iterations

Start with a fresh copy of the Part 1 master for each example. In the copy, impose $w_t\le B\hat\mu$ so that the dummy starts also satisfy (31) at the ceiling. This makes every master trial feasible for SP at $\hat\mu$. It preserves an optimum: a bounded optimal schedule at $\mu$ can use the same binaries at the ceiling by scaling its starts by $\hat\mu/\mu$. This preserves the active timing inequalities, and the existing Big-M covers the inactive rows within the ceiling.

Use the master optimum as the lower bound and retain the lowest feasible SP period as the upper bound, together with its allocation and starts. Stop at a gap of $10^{-6}$ or after 100 iterations per example.

Set `accelerated=True` to update both core points and use the ASP multipliers in (21). With `accelerated=False`, use the DSP multipliers. For either mode, take the certified master bound as $LB$ and use the verified primal SP schedule for $UB$. Keep the schedule that attains the best $UB$.

In [43]:
def solve_benders(example, accelerated=False, iteration_limit=100):
    mp = example["mp"].clone()
    z = mp.get_var_by_name(example["z"].name)
    x_mp = {key: mp.get_var_by_name(var.name) for key, var in example["x_mp"].items()}
    y_mp = {key: mp.get_var_by_name(var.name) for key, var in example["y_mp"].items()}
    for t in example["T"]:
        w = mp.get_var_by_name(example["w"][t].name)
        mp.add_constraint(w <= example["B"] * example["U"])
    mp.parameters.mip.tolerances.mipgap = 0
    mp.parameters.mip.tolerances.absmipgap = 0

    LB = 0
    UB = float("inf")
    best = None
    converged = False
    x_core = {}
    y_core = {}
    trials = []
    cuts = []
    mode = "Accelerated Benders" if accelerated else "Basic Benders"
    print(f"\n{mode} | {example['name']}")
    print(f"{'Iteration':>9} {'LB':>10} {'UB':>10} {'Gap':>10}")

    for iteration in range(1, iteration_limit + 1):
        solution = mp.solve()
        if mp.get_solve_status() != JobSolveStatus.OPTIMAL_SOLUTION:
            raise RuntimeError(f"MP status for {example['name']}: {mp.solve_details.status}")

        x_bar = read_binary_values(solution, x_mp)
        y_bar = read_binary_values(solution, y_mp)
        period, start = solve_sp(example, x_bar, y_bar)
        dual_value, pi_bar = solve_dsp(example, x_bar, y_bar)
        assert abs(period - dual_value) <= tol, "SP and DSP objectives differ"

        master_bound = mp.solve_details.best_bound
        if accelerated:
            x_core = update_core(x_core, x_bar)
            y_core = update_core(y_core, y_bar)
            pi_bar = solve_asp(example, x_bar, y_bar, x_core, y_core, dual_value)
            cuts.append(pi_bar)
        add_benders_cut(mp, example, z, x_mp, y_mp, pi_bar, solution, period)

        trial = {"period": period, "x_bar": x_bar, "y_bar": y_bar, "start": start}
        trials.append(trial)
        LB = max(LB, master_bound)
        if period < UB:
            verify_benders_schedule(example, period, x_bar, start)
            UB = period
            best = trial
        assert LB <= UB + tol, "Lower bound exceeds the best feasible period"
        gap = max(0, UB - LB)
        print(f"{iteration:9d} {LB:10.6g} {UB:10.6g} {gap:10.6g}")

        if gap <= tol:
            converged = True
            break
    result = {
        "best": best,
        "converged": converged,
        "iterations": iteration,
        "trials": trials,
        "cuts": cuts,
    }
    if converged:
        print("Optimality established within tolerance.")
    else:
        print("Iteration limit reached with an open gap.")
    return result

### Run both modes

Each mode starts from a fresh master. The table shows the certified bounds after each trial.

In [44]:
benders_results = {}
for accelerated in (False, True):
    for example in examples:
        benders_results[accelerated, example["name"]] = solve_benders(example, accelerated)


Basic Benders | Example 1 - From the paper
Iteration         LB         UB        Gap
        1          0          8          8
        2          0          5          5
        3          0          5          5
        4          0          5          5
        5          0          5          5
        6          5          5          0
Optimality established within tolerance.

Basic Benders | Example 2 - Synthetic test
Iteration         LB         UB        Gap
        1          0          3          3
        2          2          3          1
        3          2          3          1
        4          2          3          1
        5          2          3          1
        6          2          3          1
        7          2          3          1
        8          2          3          1
        9          2          3          1
       10          2          3          1
       11          3          3          0
Optimality established within tolerance.

Accelerated 

        2          0          5          5


        3          0          5          5


        4          0          5          5


        5          0          5          5


        6          5          5          0
Optimality established within tolerance.


... (27 outputs left out)

        8          2          3          1


        9          2          3          1


       10          2          3          1


       11          3          3          0
Optimality established within tolerance.


### Verify the best schedules

Check feasibility independently of the expected test periods.

In [45]:
for (accelerated, name), result in benders_results.items():
    example = next(e for e in examples if e["name"] == name)
    best = result["best"]
    period = best["period"]
    allocation = verify_benders_schedule(example, period, best["x_bar"], best["start"])
    mode = "Accelerated Benders" if accelerated else "Basic Benders"
    print(f"\n{mode} | {name}")
    print(f"Best feasible period: {period:g}")
    print(f"{'Task':<6}{'Resource':<10}{'Initial start':>14}")
    for t in example["T"]:
        print(f"{t:<6}{allocation[t]:<10}{best['start'][t]:>14g}")
    print("APSP constraints satisfied: Yes")


Basic Benders | Example 1 - From the paper
Best feasible period: 5
Task  Resource   Initial start
x     r1                     0
y     r2                     3
z     r1                    23
APSP constraints satisfied: Yes

Basic Benders | Example 2 - Synthetic test
Best feasible period: 3
Task  Resource   Initial start
A     R2                     0
B     R1                     0
APSP constraints satisfied: Yes

Accelerated Benders | Example 1 - From the paper
Best feasible period: 5
Task  Resource   Initial start
x     r1                     0
y     r2                     3
z     r1                    23
APSP constraints satisfied: Yes

Accelerated Benders | Example 2 - Synthetic test
Best feasible period: 3
Task  Resource   Initial start
A     R2                     0
B     R1                     0
APSP constraints satisfied: Yes


### Check the expected periods and cuts

Check the periods once the gap has closed. As a numerical check of cut validity, evaluate each accelerated cut at the trial decisions from both modes and compare its bound to the corresponding SP optimum. Dual feasibility gives the bound for any choice of the master variables.

In [46]:
for example in examples:
    name = example["name"]
    basic = benders_results[False, name]
    accelerated = benders_results[True, name]
    for result in (basic, accelerated):
        assert result["converged"], f"Open optimality gap for {name}"
        assert abs(result["best"]["period"] - example["expected_period"]) <= tol, (
            f"Unexpected optimal period for {name}"
        )

    trials = basic["trials"] + accelerated["trials"]
    for pi_bar in accelerated["cuts"]:
        for trial in trials:
            bound = dual_objective(example["sp"], example, pi_bar, trial["x_bar"], trial["y_bar"])
            assert bound.constant <= trial["period"] + tol, f"Cut exceeds the SP optimum for {name}"

    print(f"{name} | Basic: {basic['iterations']} iterations | Accelerated: {accelerated['iterations']} iterations")
    print("Expected periods and cross-trial cut checks passed.")

Example 1 - From the paper | Basic: 6 iterations | Accelerated: 6 iterations
Expected periods and cross-trial cut checks passed.
Example 2 - Synthetic test | Basic: 11 iterations | Accelerated: 11 iterations
Expected periods and cross-trial cut checks passed.
